<a href="https://colab.research.google.com/github/3702880330101-netizen/Smart-Cost/blob/main/SmartCost_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Instalar las librerías necesarias
# !pip install google-generativeai matplotlib requests pandas

import time
import io
import json
import requests
import pandas as pd
import matplotlib.pyplot as plt
import google.generativeai as genai
from google.colab import userdata

# ==========================================
# 1. CONFIGURACIÓN DE CREDENCIALES
# ==========================================
# Clave de API de Google AI Studio extraída de tu imagen
GEMINI_API_KEY = "AQ.Ab8RN6KycZOV613RC9wVfrdGwbC_TETgy7shEfk6hUcHHGIZA"
genai.configure(api_key=GEMINI_API_KEY)

# Token de GitHub (puedes guardarlo en Colab Secrets como 'GITHUB_TOKEN' o dejarlo opcional)
try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except:
    GITHUB_TOKEN = "" # Opcional: Reemplaza con tu token de GitHub si deseas sincronizar

# ==========================================
# 2. MOTOR DE CÁLKULO DE COSTOS (REGLAS EXACTAS)
# ==========================================
def calcular_costos(params):
    t_inicio = time.time()

    material_ajustado = params['consumo_estandar'] * (1 + (params['desperdicio'] / 100))
    costo_mp_unitario = material_ajustado * params['precio_material']
    costo_mod_unitario = params['horas_mod'] * params['tarifa_mod']
    costo_variable_unitario = costo_mp_unitario + costo_mod_unitario + params['cif_variable']
    costo_total = (costo_variable_unitario * params['volumen']) + params['costos_fijos']
    costo_unitario = costo_total / params['volumen'] if params['volumen'] > 0 else 0
    margen_contribucion_unitario = params['precio_venta'] - costo_variable_unitario

    if margen_contribucion_unitario <= 0:
        punto_equilibrio = float('inf')
        advertencia = "⚠️ ADVERTENCIA: Margen <= 0. Punto de equilibrio no válido o inalcanzable."
    else:
        punto_equilibrio = params['costos_fijos'] / margen_contribucion_unitario
        advertencia = "✅ OK (Trazable y Válido)"

    utilidad = (margen_contribucion_unitario * params['volumen']) - params['costos_fijos']

    t_fin = time.time()
    tiempo_simulacion_ms = (t_fin - t_inicio) * 1000

    return {
        'material_ajustado': material_ajustado,
        'costo_mp_unitario': costo_mp_unitario,
        'costo_mod_unitario': costo_mod_unitario,
        'costo_variable_unitario': costo_variable_unitario,
        'costo_total': costo_total,
        'costo_unitario': costo_unitario,
        'margen_contribucion_unitario': margen_contribucion_unitario,
        'punto_equilibrio': punto_equilibrio,
        'utilidad': utilidad,
        'advertencia': advertencia,
        'tiempo_simulacion_ms': tiempo_simulacion_ms
    }

# Parámetros base
base_params = {
    'material': 'Acero Inoxidable',
    'consumo_estandar': 1.0,
    'precio_material': 50.0,
    'desperdicio': 4.0,
    'horas_mod': 2.5,
    'tarifa_mod': 15.0,
    'cif_variable': 10.0,
    'costos_fijos': 5000.0,
    'volumen': 1000,
    'precio_venta': 120.0
}

# ==========================================
# 3. EJECUCIÓN DE PRUEBAS OBLIGATORIAS
# ==========================================
escenarios_prueba = {
    "Base": base_params.copy(),
    "Materia Prima +12%": {**base_params, 'precio_material': base_params['precio_material'] * 1.12},
    "Desperdicio 9%": {**base_params, 'desperdicio': 9.0},
    "Extremo (Margen <= 0)": {**base_params, 'precio_venta': 65.0, 'cif_variable': 35.0}
}

resultados_tabla = []
datos_completos_json = {}
nombres_escenarios = []
utilidades = []
costos_unitarios = []

utilidad_base = None

for nombre, params in escenarios_prueba.items():
    res = calcular_costos(params)
    nombres_escenarios.append(nombre)
    utilidades.append(res['utilidad'])
    costos_unitarios.append(res['costo_unitario'])
    datos_completos_json[nombre] = {**params, **res}

    if "Base" in nombre:
        utilidad_base = res['utilidad']
        var_utilidad = 0.0
    else:
        var_utilidad = res['utilidad'] - utilidad_base

    resultados_tabla.append({
        'Escenario': nombre,
        'Costo Unitario': f"${res['costo_unitario']:.2f}",
        'Margen Unitario': f"${res['margen_contribucion_unitario']:.2f}",
        'Utilidad Total': f"${res['utilidad']:.2f}",
        'Var. vs Base': f"${var_utilidad:.2f}",
        'Validación': res['advertencia']
    })

df_pruebas = pd.DataFrame(resultados_tabla)
print("="*90)
print("REPORTE DE PRUEBAS OBLIGATORIAS Y ESCENARIOS")
print("="*90)
display(df_pruebas)

# ==========================================
# 4. GENERACIÓN DE GRÁFICA COMPARATIVA (SVG)
# ==========================================
plt.style.use('default')
fig, ax1 = plt.subplots(figsize=(9, 5))

color = 'tab:blue'
ax1.set_ylabel('Utilidad Total ($)', color=color, fontweight='bold')
bars = ax1.bar(nombres_escenarios, utilidades, color=color, alpha=0.8, width=0.4, label='Utilidad')
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_title('Simulación Industrial: Utilidad vs Costo Unitario por Escenario', fontsize=12, fontweight='bold', pad=15)

ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Costo Unitario ($)', color=color, fontweight='bold')
ax2.plot(nombres_escenarios, costos_unitarios, color=color, marker='o', linewidth=2.5, label='Costo Unitario')
ax2.tick_params(axis='y', labelcolor=color)

fig.tight_layout()

# Guardar en buffer SVG
svg_buffer = io.BytesIO()
plt.savefig(svg_buffer, format='svg', bbox_inches='tight')
svg_buffer.seek(0)
svg_content = svg_buffer.getvalue().decode('utf-8')
plt.close()
print("\n✅ Gráfica SVG generada correctamente.")

# ==========================================
# 5. ANÁLISIS GERENCIAL CON GOOGLE GEMINI
# ==========================================
res_base = calcular_costos(base_params)

contexto_numerico = f"""
RESULTADOS CALCULADOS POR EL MOTOR (ESCENARIO BASE):
- Material analizado: {base_params['material']}
- Volumen de producción/ventas: {base_params['volumen']} unidades
- Precio de venta unitario: ${base_params['precio_venta']:.2f}
- Material ajustado: {res_base['material_ajustado']:.4f}
- Costo MP unitario: ${res_base['costo_mp_unitario']:.2f}
- Costo MOD unitario: ${res_base['costo_mod_unitario']:.2f}
- Costo variable unitario: ${res_base['costo_variable_unitario']:.2f}
- Costo total: ${res_base['costo_total']:.2f}
- Costo unitario total: ${res_base['costo_unitario']:.2f}
- Margen de contribución unitario: ${res_base['margen_contribucion_unitario']:.2f}
- Punto de equilibrio: {res_base['punto_equilibrio']:.2f} unidades
- Utilidad operativa: ${res_base['utilidad']:.2f}
"""

prompt_ia = f"""
Actúa como analista de costos industriales.
Recibirás resultados YA CALCULADOS por un simulador:
{contexto_numerico}

Tu tarea es:
1) Explicar qué indican estos resultados respecto a la sensibilidad de la utilidad, el margen y el punto de equilibrio.
2) Ordenar las variables de mayor a menor impacto cuantitativo según los datos provistos.
3) Proponer hasta 3 acciones concretas y cuantificables para probar en el simulador.
4) Señalar los supuestos y riesgos asociados.

Reglas estrictas que DEBES cumplir:
- No inventes costos, precios ni volúmenes que no estén en los datos provistos.
- No sustituyas las fórmulas matemáticas programadas en el motor.
- Toda propuesta de acción debe indicar explícitamente qué variable modificar y en qué dirección.
- Distingue claramente el resultado calculado de la recomendación analítica.
- No afirmes que una alternativa es óptima sin una comparación numérica directa.
- Haz evidentes los supuestos sobre los cuales descansan tus recomendaciones.

Formato de salida requerido:
- Diagnóstico:
- Variables críticas:
- Acciones a simular:
- Riesgos:
"""

print("\n" + "="*90)
print("DIAGNÓSTICO GERENCIAL (GENERADO POR GOOGLE GEMINI)")
print("="*90)

try:
    # Usando el modelo Gemini configurado con tu clave
    model = genai.GenerativeModel('gemini-1.5-flash')
    response = model.generate_content(prompt_ia)
    print(response.text)
except Exception as e:
    print(f"Error al generar análisis con Gemini: {e}")

# ==========================================
# 6. ENVÍO AUTOMÁTICO A GITHUB GIST
# ==========================================
if not GITHUB_TOKEN:
    print("\n⚠️ Nota: No se configuró el GITHUB_TOKEN. Los resultados y la gráfica se procesaron localmente en Colab.")
else:
    print("\n🔄 Sincronizando resultados y gráficas con GitHub Gist...")
    gist_payload = {
        "description": "Simulación de Costos Industriales - Reporte, Escenarios y Gráfica",
        "public": False,
        "files": {
            "resultados_costos.json": {
                "content": json.dumps(datos_completos_json, indent=2, default=str)
            },
            "comparativa_escenarios.svg": {
                "content": svg_content
            }
        }
    }

    headers = {
        'Authorization': f'token {GITHUB_TOKEN}',
        'Accept': 'application/vnd.github.v3+json',
        'Content-Type': 'application/json'
    }

    response = requests.post('https://api.github.com/gists', headers=headers, data=json.dumps(gist_payload))

    if response.status_code == 201:
        gist_data = response.json()
        print("¡Enviado a GitHub con éxito! 🚀")
        print(f"🔗 Enlace al Gist privado: {gist_data['html_url']}")
    else:
        print(f"❌ Error al conectar con GitHub API: {response.status_code} - {response.text}")

REPORTE DE PRUEBAS OBLIGATORIAS Y ESCENARIOS


,Escenario,Costo Unitario,Margen Unitario,Utilidad Total,Var. vs Base,Validación
0,Base,$104.50,$20.50,$15500.00,$0.00,✅ OK (Trazable y Válido)
1,Materia Prima +12%,$110.74,$14.26,$9260.00,$-6240.00,✅ OK (Trazable y Válido)
2,Desperdicio 9%,$107.00,$18.00,$13000.00,$-2500.00,✅ OK (Trazable y Válido)
3,Extremo (Margen <= 0),$129.50,$-59.50,$-64500.00,$-80000.00,⚠️ ADVERTENCIA: Margen <= 0. Punto de equilibr...



✅ Gráfica SVG generada correctamente.

DIAGNÓSTICO GERENCIAL (GENERADO POR GOOGLE GEMINI)


Error al generar análisis con Gemini: 401 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: Request had invalid authentication credentials. Expected OAuth 2 access token, login cookie or other valid authentication credential. See https://developers.google.com/identity/sign-in/web/devconsole-project.

⚠️ Nota: No se configuró el GITHUB_TOKEN. Los resultados y la gráfica se procesaron localmente en Colab.


In [ ]:
!pip install streamlit requests

In [ ]:
%%writefile app.py
import streamlit as st
import requests
import base64
import json

st.set_page_config(page_title="Conexión con GitHub", layout="centered")

st.title("Conexión con GitHub y Google Colab")

col1, col2 = st.columns(2)
with col1:
    owner = st.text_input("Usuario o Propietario de GitHub", placeholder="ej. tu-usuario")
with col2:
    repo_name = st.text_input("Nombre del Repositorio", placeholder="ej. mi-repo")

token = st.text_input("GitHub Personal Access Token (PAT)", type="password", placeholder="ghp_xxxxxxxxxxxx")
filename = st.text_input("Nombre del archivo JSON", value="costos_data.json")

if st.button("Calcular y enviar a GitHub"):
    if not token or not owner or not repo_name or not filename:
        st.error("Por favor completa todos los campos (Token, Usuario, Repositorio y Nombre de archivo).")
    else:
        try:
            datos_calculados = {
                "status": "success",
                "mensaje": "Datos de costos calculados correctamente",
                "datos": {"item_1": 150.0, "item_2": 320.5}
            }

            contenido_json = json.dumps(datos_calculados, indent=4)
            contenido_base64 = base64.b64encode(contenido_json.encode('utf-8')).decode('utf-8')

            url = f"https://api.github.com/repos/{owner}/{repo_name}/contents/{filename}"

            headers = {
                "Authorization": f"token {token}",
                "Accept": "application/vnd.github.v3+json"
            }

            response_get = requests.get(url, headers=headers)
            sha = None
            if response_get.status_code == 200:
                sha = response_get.json().get('sha')

            data = {
                "message": f"Actualización automática de {filename}",
                "content": contenido_base64
            }
            if sha:
                data["sha"] = sha

            response = requests.put(url, headers=headers, json=data)

            if response.status_code in [200, 201]:
                st.success(f"¡Archivo '{filename}' enviado y guardado exitosamente en GitHub ({owner}/{repo_name})!")
            else:
                try:
                    error_detail = response.json().get('message', response.text)
                except:
                    error_detail = response.text
                st.error(f"Error al conectar con GitHub: Error en GitHub API: {error_detail}")

        except Exception as e:
            st.error(f"Ocurrió un error inesperado: {str(e)}")

Overwriting app.py


In [1]:
import requests
import pandas as pd
import numpy as np

# ==========================================
# 1. CARGA AUTOMÁTICA DE DATOS DESDE GITHUB/GIST
# ==========================================
url_json = "https://gist.githubusercontent.com/3702880330101-netizen/3423b40e68604e988e434002708f03d9/raw/b45ef83740f163c140dac763d41016415fbd21ab/costos_data.json"

try:
    response = requests.get(url_json)
    data = response.json()
    print("¡Datos cargados exitosamente desde la web!")
except Exception as e:
    print("Aviso: No se pudo conectar a la URL, usando valores de respaldo por defecto:", e)
    data = {
        "material": "Acero Inoxidable",
        "consumo_estandar": 1.0,
        "precio_material": 50.00,
        "desperdicio": 5.0,
        "horas_mod": 2.5,
        "tarifa_mod": 15.00,
        "cif_variable": 10.00,
        "costos_fijos": 5000.00,
        "volumen": 1000,
        "precio_venta": 120.00
    }

# Mostrar los datos base en formato de tabla limpia
df_base = pd.DataFrame([data])
print("\n--- DATOS ACTUALES DE LA WEB ---")
display(df_base)


# ==========================================
# 2. MOTOR DE CÁLCULO Y SENSIBILIDAD
# ==========================================
def calcular_modelo(p):
    material_ajustado = p['consumo_estandar'] * (1 + (p['desperdicio'] / 100))
    costo_mp_unit = material_ajustado * p['precio_material']
    costo_mod_unit = p['horas_mod'] * p['tarifa_mod']
    costo_var_unit = costo_mp_unit + costo_mod_unit + p['cif_variable']
    costo_total = (costo_var_unit * p['volumen']) + p['costos_fijos']
    costo_unitario = costo_total / p['volumen'] if p['volumen'] > 0 else 0
    margen_contrib_unit = p['precio_venta'] - costo_var_unit
    punto_equilibrio = np.ceil(p['costos_fijos'] / margen_contrib_unit) if margen_contrib_unit > 0 else 0
    utilidad = (margen_contrib_unit * p['volumen']) - p['costos_fijos']

    return {
        "Costo Unitario": costo_unitario,
        "Costo Variable Unitario": costo_var_unit,
        "Margen de Contribución": margen_contrib_unit,
        "Punto de Equilibrio (u.)": punto_equilibrio,
        "Utilidad Total": utilidad
    }

res_base = calcular_modelo(data)
print("\n--- RESULTADOS DEL ESCENARIO BASE ---")
for k, v in res_base.items():
    print(f"• {k}: {v:,.2f}" if isinstance(v, float) else f"• {k}: {v}")


# ==========================================
# 3. PRIORIZACIÓN DE VARIABLES (IMPACTO CUANTITATIVO)
# ==========================================
variables_a_probar = ['precio_venta', 'precio_material', 'volumen', 'desperdicio', 'horas_mod', 'costos_fijos']
impactos = []
utilidad_base = res_base['Utilidad Total']

for var in variables_a_probar:
    data_mod = data.copy()
    data_mod[var] = data_mod[var] * 1.10
    res_mod = calcular_modelo(data_mod)
    dif_utilidad = res_mod['Utilidad Total'] - utilidad_base
    impactos.append({
        "Variable": var,
        "Impacto en Utilidad por +10% ($)": dif_utilidad
    })

df_impactos = pd.DataFrame(impactos).sort_values(by="Impacto en Utilidad por +10% ($)", ascending=False)
print("\n--- PRIORIZACIÓN DE VARIABLES (Impacto en Utilidad) ---")
display(df_impactos)


# ==========================================
# 4. SIMULACIÓN DE ESCENARIOS DE MEJORA
# ==========================================
escenarios = {
    "Base Actual": data.copy(),
    "Reducción de Desperdicio (2%)": {**data, "desperdicio": 2.0},
    "Optimización de MP (-5% Precio)": {**data, "precio_material": data["precio_material"] * 0.95},
    "Aumento de Volumen (+5%)": {**data, "volumen": int(data["volumen"] * 1.05)}
}

resultados_escenarios = []
for nombre, params in escenarios.items():
    res = calcular_modelo(params)
    resultados_escenarios.append({
        "Escenario": nombre,
        "Costo Unitario ($)": res["Costo Unitario"],
        "Punto de Equilibrio": res["Punto de Equilibrio (u.)"],
        "Utilidad Total ($)": res["Utilidad Total"]
    })

df_escenarios = pd.DataFrame(resultados_escenarios)
print("\n--- COMPARATIVA DE ESCENARIOS DE MEJORA ---")
display(df_escenarios)


# ==========================================
# 5. REDACCIÓN DE CONCLUSIONES GERENCIALES
# ==========================================
mejor_escenario = df_escenarios.loc[df_escenarios["Utilidad Total ($)"].idxmax()]
variable_clave = df_impactos.iloc[0]["Variable"]

print("\n" + "=".center(60, "="))
print(" REPORTE Y CONCLUSIONES GERENCIALES AUTOMATIZADAS ".center(60, "="))
print("=".center(60, "="))
print(f"1. Sensibilidad y Prioridad: La variable que ejerce mayor influencia cuantitativa en el resultado financiero es '{variable_clave}'. Cualquier ajuste o negociación debe centrarse principalmente en su control.")
print(f"2. Umbral Operativo: La empresa requiere comercializar un mínimo de {res_base['Punto de Equilibrio (u.)']:,.0f} unidades para cubrir los costos fijos actuales de ${data['costos_fijos']:,.2f}.")
print(f"3. Rendimiento Óptimo: El mejor escenario proyectado es '{mejor_escenario['Escenario']}', logrando una utilidad total de ${mejor_escenario['Utilidad Total ($)']:,.2f}.")
print(f"4. Recomendación Estratégica: Se aconseja aplicar formalmente las iniciativas de mejora simuladas (como la optimización de insumos y control de mermas) y actualizar de forma continua los parámetros desde el formulario web conectado a GitHub.")
print("="*60)

¡Datos cargados exitosamente desde la web!

--- DATOS ACTUALES DE LA WEB ---


,material,consumo_estandar,precio_material,desperdicio,horas_mod,tarifa_mod,cif_variable,costos_fijos,volumen,precio_venta,timestamp
0,Acero Inoxidable,1.0,50.0,5.0,2.5,15.0,10.0,5000.0,1000,120.0,2026-10-07T01:42:00.000Z



--- RESULTADOS DEL ESCENARIO BASE ---
• Costo Unitario: 105.00
• Costo Variable Unitario: 100.00
• Margen de Contribución: 20.00
• Punto de Equilibrio (u.): 250.00
• Utilidad Total: 15,000.00

--- PRIORIZACIÓN DE VARIABLES (Impacto en Utilidad) ---


,Variable,Impacto en Utilidad por +10% ($)
0,precio_venta,12000.0
2,volumen,2000.0
3,desperdicio,-250.0
5,costos_fijos,-500.0
4,horas_mod,-3750.0
1,precio_material,-5250.0



--- COMPARATIVA DE ESCENARIOS DE MEJORA ---


,Escenario,Costo Unitario ($),Punto de Equilibrio,Utilidad Total ($)
0,Base Actual,105.000000,250.0,15000.0
1,Reducción de Desperdicio (2%),103.500000,233.0,16500.0
2,Optimización de MP (-5% Precio),102.375000,221.0,17625.0
3,Aumento de Volumen (+5%),104.761905,250.0,16000.0



===== REPORTE Y CONCLUSIONES GERENCIALES AUTOMATIZADAS =====
1. Sensibilidad y Prioridad: La variable que ejerce mayor influencia cuantitativa en el resultado financiero es 'precio_venta'. Cualquier ajuste o negociación debe centrarse principalmente en su control.
2. Umbral Operativo: La empresa requiere comercializar un mínimo de 250 unidades para cubrir los costos fijos actuales de $5,000.00.
3. Rendimiento Óptimo: El mejor escenario proyectado es 'Optimización de MP (-5% Precio)', logrando una utilidad total de $17,625.00.
4. Recomendación Estratégica: Se aconseja aplicar formalmente las iniciativas de mejora simuladas (como la optimización de insumos y control de mermas) y actualizar de forma continua los parámetros desde el formulario web conectado a GitHub.
